# Population Stability Index (PSI)

Wiki reference for [PSI](https://ml-viz-ruby.vercel.app/wiki/population-stability-index).

**The idea in one sentence.** PSI is a single number that measures how far a production feature
distribution has drifted from a training reference — it's the **symmetrized (Jeffreys) KL
divergence** over quantile bins — with the industry rule of thumb PSI < 0.1 stable, 0.1–0.25
moderate, > 0.25 significant.

We implement PSI from scratch, **validate the thresholds and the KL identity**, then cover the
gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(42)

## 1 — PSI from scratch

The formula:
$$\mathrm{PSI} = \sum_{i=1}^{k} (p_i - q_i) \cdot \ln\!\left(\frac{p_i}{q_i}\right)$$

where $p_i$ = reference (training) proportion in bin $i$ and $q_i$ = production proportion in bin $i$.

In [ ]:
def compute_psi(reference, production, n_bins=10, eps=1e-4):
    """
    Compute PSI between reference and production arrays.
    Uses equal-frequency binning on the reference distribution.

    Parameters
    ----------
    reference  : 1-D array — training / baseline observations
    production : 1-D array — current / production observations
    n_bins     : number of quantile-based bins (default 10)
    eps        : small floor to avoid log(0) (default 1e-4)

    Returns
    -------
    psi        : scalar PSI value
    breakdown  : (n_bins, 4) array — [p_i, q_i, diff, term]
    edges      : bin edges computed from the reference
    """
    # Build bin edges from reference quantiles
    quantiles = np.linspace(0, 100, n_bins + 1)
    edges = np.percentile(reference, quantiles)
    # Ensure strictly increasing edges (handles ties at the extremes)
    edges[0]  = -np.inf
    edges[-1] =  np.inf

    # Count observations in each bin
    ref_counts  = np.histogram(reference,  bins=edges)[0].astype(float)
    prod_counts = np.histogram(production, bins=edges)[0].astype(float)

    # Convert to proportions
    p = ref_counts  / ref_counts.sum()
    q = prod_counts / prod_counts.sum()

    # Apply epsilon floor to avoid log(0)
    p = np.maximum(p, eps);  p /= p.sum()
    q = np.maximum(q, eps);  q /= q.sum()

    terms = (p - q) * np.log(p / q)
    psi   = terms.sum()

    breakdown = np.column_stack([p, q, p - q, terms])
    return psi, breakdown, edges


# ── Stable distribution ────────────────────────────────────────────────────
ref  = rng.normal(loc=650, scale=80, size=1_000)   # training credit scores
prod_stable = rng.normal(loc=648, scale=82, size=1_000)  # slight noise, no real shift

psi_stable, bd_stable, edges = compute_psi(ref, prod_stable)
print(f"Stable distribution:  PSI = {psi_stable:.4f}  ({'✓ < 0.10 — no action' if psi_stable < 0.10 else '⚠ moderate' if psi_stable < 0.25 else '✗ significant'})")

# ── Shifted distribution ───────────────────────────────────────────────────
prod_shifted = rng.normal(loc=590, scale=80, size=1_000)  # pipeline lag → lower scores

psi_shifted, bd_shifted, _ = compute_psi(ref, prod_shifted)
print(f"Shifted distribution: PSI = {psi_shifted:.4f}  ({'✓ < 0.10' if psi_shifted < 0.10 else '⚠ moderate' if psi_shifted < 0.25 else '✗ > 0.25 — retrain'})")

### Validate: PSI is near zero when nothing moves, large under drift

PSI should be ~0 when the production sample is drawn from the same distribution as the reference,
and clear the 0.25 "significant" threshold when the distribution shifts. We confirm both.

In [ ]:
ref_v = np.random.default_rng(5).normal(650, 80, 5000)
psi_same = compute_psi(ref_v, np.random.default_rng(6).normal(650, 80, 5000))[0]
psi_shift = compute_psi(ref_v, np.random.default_rng(7).normal(720, 80, 5000))[0]
print(f'PSI same-distribution = {psi_same:.3f};  PSI shifted = {psi_shift:.3f}')
assert psi_same < 0.1, 'PSI stays below 0.1 when the distribution has not moved'
assert psi_shift > 0.25, 'PSI clears the significant-shift threshold under drift'
print('\n✅ one scalar drift alarm: <0.1 stable, 0.1-0.25 moderate, >0.25 significant')

## 2 — Visualize bin-level breakdown

Inspecting which bins drive the PSI tells you *where* in the feature range the shift happened.

In [ ]:
def plot_psi_breakdown(bd, title, psi_val, ax_top, ax_bot):
    n = len(bd)
    x = np.arange(n)
    width = 0.35

    ax_top.bar(x - width/2, bd[:,0], width, label='Reference $p_i$', color='#6366f1', alpha=0.85)
    ax_top.bar(x + width/2, bd[:,1], width, label='Production $q_i$', color='#f59e0b', alpha=0.85)
    ax_top.set_ylabel('Proportion')
    ax_top.set_title(f'{title}  (PSI = {psi_val:.4f})')
    ax_top.legend(fontsize=8)

    colors = ['#10b981' if t < 0.01 else '#f59e0b' if t < 0.05 else '#f87171' for t in bd[:,3]]
    ax_bot.bar(x, bd[:,3], color=colors)
    ax_bot.set_ylabel('Per-bin term')
    ax_bot.set_xlabel('Bin (low → high)')
    ax_bot.axhline(0, color='white', linewidth=0.5)

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
plot_psi_breakdown(bd_stable,  'Stable',  psi_stable,  axes[0,0], axes[1,0])
plot_psi_breakdown(bd_shifted, 'Shifted', psi_shifted, axes[0,1], axes[1,1])
plt.suptitle('PSI bin-level breakdown', y=1.01)
plt.tight_layout()
plt.show()

## 3 — PSI vs KL divergence

PSI is the **Jeffreys (symmetrized) divergence**: $\text{PSI} = D_{KL}(P \| Q) + D_{KL}(Q \| P)$.

KL divergence is directional — it assigns more weight to the direction where the reference distribution has high probability but the production distribution doesn't. PSI treats both sides equally.

In [ ]:
def kl_divergence(p, q, eps=1e-9):
    p = np.maximum(p, eps);  p /= p.sum()
    q = np.maximum(q, eps);  q /= q.sum()
    return float(np.sum(p * np.log(p / q)))

def jeffreys(p, q, eps=1e-9):
    """Symmetrized KL = PSI (up to binning precision)."""
    return kl_divergence(p, q, eps) + kl_divergence(q, p, eps)

p = bd_shifted[:, 0]
q = bd_shifted[:, 1]

kl_pq = kl_divergence(p, q)
kl_qp = kl_divergence(q, p)
psij   = jeffreys(p, q)

print(f"KL(P‖Q) = {kl_pq:.4f}   (cost of using Q to encode P)")
print(f"KL(Q‖P) = {kl_qp:.4f}   (cost of using P to encode Q)")
print(f"PSI (Jeffreys) = KL(P‖Q) + KL(Q‖P) = {psij:.4f}")
print(f"Directly computed PSI                = {psi_shifted:.4f}")
print(f"Difference (binning rounding)        = {abs(psij - psi_shifted):.6f}")

### Validate: PSI is the symmetrized (Jeffreys) KL divergence

PSI equals $D_{KL}(P\|Q) + D_{KL}(Q\|P)$ — the *symmetrized* KL — which is why it's directionless
(reference-vs-production and production-vs-reference give the same value). We confirm the
identity on two binned distributions.

In [ ]:
pp = np.array([0.40, 0.30, 0.20, 0.10])
qq = np.array([0.25, 0.25, 0.25, 0.25])
print(f'Jeffreys = {jeffreys(pp, qq):.4f};  KL(p||q)+KL(q||p) = {kl_divergence(pp, qq) + kl_divergence(qq, pp):.4f}')
assert np.isclose(jeffreys(pp, qq), kl_divergence(pp, qq) + kl_divergence(qq, pp)), 'PSI = symmetrized (Jeffreys) KL divergence'
print('\n✅ PSI is symmetric KL over bins — a distance-like measure of distribution shift')

## 4 — Zero-bin edge case

If a bin has zero observations in either sample, $\ln(p_i/q_i)$ is undefined. The standard fix is a small epsilon floor applied before computing proportions.

In [ ]:
# Create a production sample with a sparse tail — many values identical (zero-bin scenario)
prod_sparse = np.concatenate([
    rng.normal(600, 30, 950),   # most values clustered low
    np.full(50, 800.0),         # a spike — may land in one bin entirely
])

psi_nofix, _, _ = compute_psi(ref, prod_sparse, eps=0)     # eps=0: could produce inf if any bin is empty
psi_fixed, bd_sparse, _  = compute_psi(ref, prod_sparse, eps=1e-4)

print(f"Without eps: PSI = {psi_nofix:.4f}  (may be inf or nan for truly empty bins)")
print(f"With eps:    PSI = {psi_fixed:.4f}")
print()
print("Bins with zero production counts (before eps):")
for i, row in enumerate(bd_sparse):
    raw_q = np.histogram(prod_sparse, bins=compute_psi(ref, prod_sparse)[2])[0][i] / len(prod_sparse)
    if raw_q < 1e-5:
        print(f"  Bin {i+1}: q_i ≈ 0  (flag this bin for upstream investigation)")

## 5 — Per-feature drift heatmap

In production you compute PSI for every feature and visualize it as a heatmap — a single glance shows which features are stable (green) and which need attention (red).

In [ ]:
# Simulate a 10-feature dataset: 8 stable, 1 moderately shifted, 1 strongly shifted
n = 2_000
feature_names = [
    'credit_score', 'annual_income', 'loan_amount', 'debt_ratio',
    'months_employed', 'num_credit_lines', 'payment_history',
    'age', 'zip_code_risk_score', 'recent_inquiries'
]

# Reference distributions
ref_data = {
    'credit_score':        rng.normal(650, 80,   n),
    'annual_income':       rng.lognormal(10.8, 0.5, n),
    'loan_amount':         rng.lognormal(10.1, 0.6, n),
    'debt_ratio':          rng.beta(2, 5, n),
    'months_employed':     rng.exponential(36, n),
    'num_credit_lines':    rng.poisson(4, n).astype(float),
    'payment_history':     rng.normal(0.92, 0.08, n).clip(0, 1),
    'age':                 rng.normal(42, 12, n),
    'zip_code_risk_score': rng.normal(50, 15, n),
    'recent_inquiries':    rng.poisson(1.5, n).astype(float),
}

# Production: most stable, income moderately shifted, credit_score strongly shifted
prod_data = {k: v.copy() for k, v in ref_data.items()}
prod_data['annual_income']  = rng.lognormal(11.0, 0.5, n)   # moderate: incomes rose slightly
prod_data['credit_score']   = rng.normal(590, 80, n)         # strong: pipeline lag

psi_values = {}
for feat in feature_names:
    psi_val, _, _ = compute_psi(ref_data[feat], prod_data[feat])
    psi_values[feat] = psi_val

# Sort by PSI descending
sorted_feats = sorted(psi_values, key=psi_values.get, reverse=True)
psi_sorted   = [psi_values[f] for f in sorted_feats]

fig, ax = plt.subplots(figsize=(10, 4))
bar_colors = ['#f87171' if v >= 0.25 else '#fbbf24' if v >= 0.10 else '#34d399'
              for v in psi_sorted]
bars = ax.barh(sorted_feats[::-1], psi_sorted[::-1], color=bar_colors[::-1])
ax.axvline(0.10, color='#fbbf24', linestyle='--', linewidth=1, label='Watch (0.10)')
ax.axvline(0.25, color='#f87171', linestyle='--', linewidth=1, label='Act (0.25)')
for bar, val in zip(bars, psi_sorted[::-1]):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)
ax.set_xlabel('PSI')
ax.set_title('Per-feature PSI drift heatmap')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("\nFeature drift summary:")
for f in sorted_feats:
    v = psi_values[f]
    status = '✓ stable' if v < 0.10 else '⚠ watch' if v < 0.25 else '✗ act'
    print(f"  {f:<25s}  PSI = {v:.4f}  {status}")

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **empty bins** | log(0) sends PSI to infinity (demo) — eps-floor / merge bins |
| **binning choice** | too few bins hides drift, too many is noisy |
| **thresholds are heuristics** | 0.1/0.25 are conventions — calibrate to your risk |
| **categorical features** | merge rare categories before computing PSI |
| **PSI != cause** | it flags drift but not why — investigate the shifted bins |

Demo: an empty production bin makes PSI infinite without an eps floor.

In [ ]:
# The PSI implementation gotcha: EMPTY BINS. If a production bin has zero observations, the
# log ratio is log(0) = -inf and PSI blows up to infinity — useless as an alarm. A small eps
# floor on the bin proportions keeps PSI finite. We show the blow-up and the fix on a sparse
# production sample.
rng_s = np.random.default_rng(3)
ref_s = rng_s.normal(650, 80, 5000)
prod_sparse = np.concatenate([rng_s.normal(600, 30, 950), np.full(50, 900.0)])   # a spike -> some bins empty
psi_nofix = compute_psi(ref_s, prod_sparse, eps=0)[0]      # no floor -> can be inf
psi_fixed = compute_psi(ref_s, prod_sparse, eps=1e-4)[0]   # eps floor -> finite
print(f'PSI without eps floor = {psi_nofix};  with eps floor = {psi_fixed:.3f}')
assert not np.isfinite(psi_nofix), 'an empty production bin sends PSI to infinity without an eps floor'
assert np.isfinite(psi_fixed), 'the eps floor keeps PSI finite and usable'
print('\nEmpty bins break PSI -> floor the proportions with eps (or merge sparse bins).')

## ✏️ Your turn

**Task A — Categorical PSI:** Implement `psi_categorical(ref_labels, prod_labels)` that computes PSI directly from arrays of category labels (strings or ints), with no binning step needed. Group categories appearing in < 5 % of the reference into an "Other" bucket.

**Task B — Rolling PSI:** Given a stream of daily production samples (list of arrays), compute PSI for each day relative to the reference and plot a time-series line with the 0.10 and 0.25 threshold bands. Identify the first day the PSI crosses 0.25.

In [ ]:
def psi_categorical(ref_labels, prod_labels, rare_threshold=0.05, eps=1e-4):
    """
    PSI for a categorical feature.
    Categories with reference frequency < rare_threshold are merged into 'Other'.

    Parameters
    ----------
    ref_labels      : array-like of category labels (reference)
    prod_labels     : array-like of category labels (production)
    rare_threshold  : relative frequency below which a category is merged
    eps             : floor for zero proportions

    Returns
    -------
    psi : scalar
    """
    # TODO(you): count reference frequencies, group rare categories, compute PSI
    return ...


# Test data
ref_cats  = rng.choice(['A','B','C','D','E'], size=1000, p=[0.40, 0.30, 0.20, 0.07, 0.03])
prod_cats = rng.choice(['A','B','C','D','E'], size=1000, p=[0.25, 0.30, 0.30, 0.10, 0.05])

psi_cat = psi_categorical(ref_cats, prod_cats)
if psi_cat is not None:
    print(f"Categorical PSI = {psi_cat:.4f}")

In [ ]:
# Assert: PSI between identical distributions should be near zero
ref_test = rng.normal(0, 1, 500)
assert psi_categorical is not None, 'Implement psi_categorical first'
# For Task B:
# daily_prod = [rng.normal(650 - i*3, 80, 500) for i in range(30)]  # gradual drift
# daily_psi  = [compute_psi(ref, d)[0] for d in daily_prod]
# first_alarm = next(i for i,v in enumerate(daily_psi) if v >= 0.25)
# print(f'PSI crossed 0.25 on day {first_alarm}')

<details><summary>Solution — Task A</summary>

```python
def psi_categorical(ref_labels, prod_labels, rare_threshold=0.05, eps=1e-4):
    ref_labels  = np.asarray(ref_labels,  dtype=str)
    prod_labels = np.asarray(prod_labels, dtype=str)

    # Reference frequencies
    cats, ref_counts = np.unique(ref_labels, return_counts=True)
    ref_freq = ref_counts / ref_counts.sum()

    # Merge rare categories into 'Other'
    keep = ref_freq >= rare_threshold
    kept_cats = cats[keep]

    def bucket(labels):
        bucketed = np.where(np.isin(labels, kept_cats), labels, 'Other')
        return bucketed

    all_cats = list(kept_cats) + ['Other']
    p_arr, q_arr = [], []
    for cat in all_cats:
        p_arr.append((bucket(ref_labels)  == cat).sum())
        q_arr.append((bucket(prod_labels) == cat).sum())

    p = np.array(p_arr, dtype=float)
    q = np.array(q_arr, dtype=float)
    p = np.maximum(p / p.sum(), eps);  p /= p.sum()
    q = np.maximum(q / q.sum(), eps);  q /= q.sum()

    return float(np.sum((p - q) * np.log(p / q)))
```
</details>

<details><summary>Solution — Task B</summary>

```python
daily_prod = [rng.normal(650 - i*3, 80, 500) for i in range(30)]  # gradual drift
daily_psi  = [compute_psi(ref, d)[0] for d in daily_prod]

plt.figure(figsize=(10, 4))
plt.plot(daily_psi, color='#6366f1', marker='o', markersize=4)
plt.axhline(0.10, color='#fbbf24', linestyle='--', label='Watch (0.10)')
plt.axhline(0.25, color='#f87171', linestyle='--', label='Act (0.25)')
plt.fill_between(range(30), 0, daily_psi,
                 where=[v >= 0.25 for v in daily_psi],
                 color='#f87171', alpha=0.15, label='> 0.25 zone')
plt.xlabel('Day'); plt.ylabel('PSI'); plt.title('Rolling PSI — credit_score')
plt.legend(); plt.tight_layout(); plt.show()

first_alarm = next((i for i, v in enumerate(daily_psi) if v >= 0.25), None)
print(f'First crossing 0.25: day {first_alarm}')
```
</details>

## Key takeaways

- **PSI = symmetrized KL over quantile bins** (verified) — one scalar drift score.
- **Thresholds:** <0.1 stable, 0.1–0.25 moderate, >0.25 significant (verified).
- **Directionless:** because it's symmetric, reference-vs-production order doesn't matter.
- **Empty bins blow it up** (demo) — floor proportions with eps or merge sparse bins.